# vLLM serve v1: một model, API kiểu OpenAI, ép JSON theo schema

Cách chạy: Settings → Accelerator **GPU T4 x2**, Internet **On**. Add-ons → Secrets:
`HF_TOKEN` (bắt buộc với Llama/Gemma), `VLLM_API_KEY` (tự đặt, dài ≥ 32 ký tự),
`NTFY_TOPIC` (tùy chọn). Sửa `MODEL_KEY` ở ô dưới rồi **Run All**.

URL in ra ở cuối, không kèm key. Ở máy local:
`export VLLM_BASE_URL=https://<...>.trycloudflare.com/v1` và `export VLLM_API_KEY=...`.
Server tự tắt sau `IDLE_MINUTES` phút không có request. **Xong việc nhớ bấm Stop session** để
không bị trừ quota GPU.

In [ ]:
# ruff: noqa: E401, E501, E702, B904, I001, S101, S103, S310, S603, S607
# Kaggle launcher: shells out and calls localhost/tunnel URLs cell by cell.
MODEL_KEY = "qwen3_8b"      # qwen3_8b | llama31_8b | gemma4_12b
VLLM_VERSION = "0.32.0"     # pin 2026-10-25; None = bản stable mới nhất
IDLE_MINUTES = 30
MAX_HOURS = 11.5
ENFORCE_EAGER = False       # bật True nếu thiếu VRAM lúc capture CUDA graph
DISCOVERY = None            # launcher điền {"topic", "public_key_pem"} khi chạy nền qua API
HF_TOKEN_INLINE = None      # launcher có thể điền khi notebook không có Secret HF_TOKEN; không commit

MODELS = {
    "qwen3_8b":   {"repo": "Qwen/Qwen3-8B", "revision": "b968826d9c46dd6066d109eabc6255188de91218", "tp": 2, "max_len": 8192, "nightly": False},
    "llama31_8b": {"repo": "meta-llama/Llama-3.1-8B-Instruct", "revision": "0e9e39f249a16976918f6564b8830bc894c89659", "tp": 2, "max_len": 8192, "nightly": False},
    "gemma4_12b": {"repo": "google/gemma-4-12B-it", "revision": None, "tp": 2, "max_len": 8192, "nightly": True},
}
SPEC = MODELS[MODEL_KEY]
REVISION = SPEC["revision"]  # None ở lần chạy đầu; pin SHA in ra vào MODELS và configs/models/backbones_v2.yaml
SEED, PORT = 42, 8000

In [ ]:
import base64, json, os, re, secrets as pysecrets, subprocess, sys, time, uuid, urllib.request
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def secret(name, required=True):
    try:
        return secrets.get_secret(name)
    except Exception:
        if required:
            raise SystemExit(f"Thiếu Kaggle Secret {name}")
        return None

# Chạy nền qua API: không có Secret thì tự sinh key và chỉ gửi bản mã hóa qua DISCOVERY.
API_KEY = secret("VLLM_API_KEY", required=DISCOVERY is None) or pysecrets.token_urlsafe(40)
assert len(API_KEY) >= 32, "VLLM_API_KEY phải dài ít nhất 32 ký tự"
HF_TOKEN = secret("HF_TOKEN", required=False) or HF_TOKEN_INLINE
NTFY_TOPIC = secret("NTFY_TOPIC", required=False)
BOOT_ID = uuid.uuid4().hex[:12]
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
GPUS = int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU "))
TP = min(SPEC["tp"], GPUS)
print("boot", BOOT_ID, "gpus", GPUS, "tensor_parallel", TP)

In [ ]:
if SPEC["nightly"]:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", "--pre", "vllm",
           "--extra-index-url", "https://wheels.vllm.ai/nightly"]
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"]
subprocess.run(cmd, check=True)
# vLLM kéo torch bản CUDA khác torchaudio có sẵn của Kaggle; model text không cần torchaudio.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=True)
VERSIONS = {pkg: subprocess.run([sys.executable, "-c", f"import {pkg}; print({pkg}.__version__)"],
                                capture_output=True, text=True).stdout.strip()
            for pkg in ("vllm", "torch", "transformers")}
print(VERSIONS)

In [ ]:
from huggingface_hub import HfApi, snapshot_download

info = HfApi(token=HF_TOKEN).model_info(SPEC["repo"], revision=REVISION)
RESOLVED = info.sha
if REVISION and RESOLVED != REVISION:
    raise SystemExit(f"Revision mismatch {RESOLVED} != {REVISION}")
MODEL_DIR = snapshot_download(SPEC["repo"], revision=RESOLVED, token=HF_TOKEN,
                              allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "*.jinja"])
SERVED = f"{SPEC['repo']}@{RESOLVED}"
print("served model name:", SERVED)
if not REVISION:
    print("LẦN ĐẦU: dán SHA này vào REVISION và configs/models/backbones_v2.yaml ->", RESOLVED)

In [ ]:
log = open("/kaggle/working/vllm.log", "w")
args = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_DIR,
        "--served-model-name", SERVED, "--api-key", API_KEY, "--port", str(PORT),
        "--dtype", "half", "--tensor-parallel-size", str(TP), "--max-model-len", str(SPEC["max_len"]),
        "--gpu-memory-utilization", "0.90", "--seed", str(SEED), "--generation-config", "vllm"]
if ENFORCE_EAGER:
    args.append("--enforce-eager")
server = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)

deadline = time.time() + 30 * 60
while time.time() < deadline:
    if server.poll() is not None:
        print(open("/kaggle/working/vllm.log").read()[-4000:])
        raise SystemExit("vLLM thoát khi đang khởi động; xem log ở trên")
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
        break
    except Exception:
        time.sleep(10)
else:
    raise SystemExit("vLLM không lên sau 30 phút")
print("vLLM healthy")

In [ ]:
TOOLS = ["doc_search", "doc_read", "db_query", "calculator", "cached_search", "cached_fetch",
         "send_email_mock", "post_webhook_mock"]
def obj(props):
    return {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}
AGENT_SCHEMA = {"anyOf": [
    obj({"action": obj({"name": {"type": "string", "enum": TOOLS}, "arguments": {"type": "object"}})}),
    obj({"final_answer": obj({"answer": {"type": "string", "minLength": 1}})})]}

def chat(base, messages, max_tokens=256):
    extra = {"chat_template_kwargs": {"enable_thinking": False}} if "Qwen3" in SPEC["repo"] else {}
    body = {"model": SERVED, "messages": messages, "temperature": 0, "seed": SEED,
            "max_tokens": max_tokens, "request_id": "warmup-" + uuid.uuid4().hex,
            "response_format": {"type": "json_schema",
                                "json_schema": {"name": "agent_turn", "schema": AGENT_SCHEMA, "strict": True}},
            **extra}
    req = urllib.request.Request(base + "/v1/chat/completions", data=json.dumps(body).encode(),
        headers={"Authorization": "Bearer " + API_KEY, "Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=180))

t = time.time()
r = chat(f"http://127.0.0.1:{PORT}", [
    {"role": "system", "content": "Bạn là trợ lý. Chỉ trả về một JSON object."},
    {"role": "user", "content": "Hãy tính 12 * 7 bằng công cụ calculator."}])
text = r["choices"][0]["message"]["content"]
json.loads(text)
assert r["choices"][0]["finish_reason"] == "stop", r["choices"][0]["finish_reason"]
print(f"warmup OK {time.time()-t:.1f}s:", text, r.get("usage"))

In [ ]:
subprocess.run(["wget", "-q", "-O", "/kaggle/working/cloudflared",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
               check=True)
os.chmod("/kaggle/working/cloudflared", 0o755)
CF_VERSION = subprocess.run(["/kaggle/working/cloudflared", "--version"], capture_output=True,
                            text=True).stdout.strip()
tunnel_log = open("/kaggle/working/tunnel.log", "w")
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url",
                           f"http://127.0.0.1:{PORT}"], stdout=tunnel_log, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    if found:
        URL = found[0]
        break
    time.sleep(2)
assert URL, "Không lấy được URL tunnel"
for _ in range(30):  # DNS of a new quick tunnel can lag
    try:
        req = urllib.request.Request(URL + "/v1/models", headers={"Authorization": "Bearer " + API_KEY})
        ids = [m["id"] for m in json.load(urllib.request.urlopen(req, timeout=20))["data"]]
        break
    except Exception:
        time.sleep(5)
else:
    raise SystemExit("Tunnel chưa truy cập được")
assert ids == [SERVED], ids
ENDPOINT = {"boot_id": BOOT_ID, "base_url": URL + "/v1", "model_key": MODEL_KEY, "repo": SPEC["repo"],
            "revision": RESOLVED, "served_model_name": SERVED, "versions": VERSIONS,
            "cloudflared": CF_VERSION, "tensor_parallel": TP, "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
json.dump(ENDPOINT, open("/kaggle/working/endpoint.json", "w"), indent=2)
print(json.dumps(ENDPOINT, indent=2))
print(f"\nexport VLLM_BASE_URL={URL}/v1")
if DISCOVERY:
    from cryptography.hazmat.primitives import hashes, serialization
    from cryptography.hazmat.primitives.asymmetric import padding
    public = serialization.load_pem_public_key(DISCOVERY["public_key_pem"].encode())
    sealed = public.encrypt(json.dumps({"base_url": URL + "/v1", "api_key": API_KEY, "boot_id": BOOT_ID,
                                        "served_model_name": SERVED}).encode(),
                            padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()),
                                         algorithm=hashes.SHA256(), label=None))
    urllib.request.urlopen(urllib.request.Request("https://ntfy.sh/" + DISCOVERY["topic"],
        data=json.dumps({"boot_id": BOOT_ID, "ciphertext": base64.b64encode(sealed).decode()}).encode()),
        timeout=20)
    print("đã gửi discovery đã mã hóa")
if NTFY_TOPIC:
    urllib.request.urlopen(urllib.request.Request("https://ntfy.sh/" + NTFY_TOPIC,
        data=f"READY {MODEL_KEY} {URL}/v1 boot={BOOT_ID}".encode()))

In [ ]:
def served_requests():
    text = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/metrics", timeout=10).read().decode()
    return sum(float(line.rsplit(" ", 1)[1]) for line in text.splitlines()
               if line.startswith("vllm:request_success_total"))

start, last_change, last_count = time.time(), time.time(), served_requests()
try:
    while True:
        time.sleep(60)
        if server.poll() is not None or tunnel.poll() is not None:
            print("vLLM hoặc tunnel đã dừng; xem /kaggle/working/*.log")
            break
        count = served_requests()
        if count != last_count:
            last_count, last_change = count, time.time()
        idle = (time.time() - last_change) / 60
        if int(time.time() - start) % 300 < 60:
            print(time.strftime("%H:%M"), f"requests={int(count)} idle={idle:.0f}m")
        if idle >= IDLE_MINUTES or time.time() - start >= MAX_HOURS * 3600:
            print("Tự tắt: hết thời gian chờ hoặc chạm giới hạn giờ")
            break
finally:
    tunnel.terminate(); server.terminate()
    print("Đã tắt server. Hãy bấm Stop session để trả GPU.")